# MA3632 — Workshop 7: Logistic Regression and Classification

This workshop accompanies Lecture 7. Parts A and B implement logistic regression
from scratch and study decision boundaries. Part C examines regularisation. Part D
covers the full suite of classification performance metrics. Part E builds ROC curves
and compares models. Part F extends to multiclass problems via softmax.

Work through all parts in order. Take-home exercises are at the end.

## Setup

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import make_classification, make_moons, load_wine, load_iris
from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.dummy import DummyClassifier
from sklearn.pipeline import Pipeline
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, roc_curve, roc_auc_score,
    precision_recall_curve, classification_report
)
import warnings
warnings.filterwarnings("ignore")

rng = np.random.default_rng(0)

# ── Load Wine dataset once; binarise for Parts C–E ──────────────────────────
# Class 0 (Barolo cultivar, 59 samples) vs classes 1+2 (119 samples).
wine = load_wine(as_frame=True)
X_wn = wine.data.values
y_wn = (wine.target.values == 0).astype(int)   # 1 = Barolo, 0 = other

X_tr_wn, X_te_wn, y_tr_wn, y_te_wn = train_test_split(
    X_wn, y_wn, test_size=0.3, random_state=0, stratify=y_wn
)
sc_wn = StandardScaler()
X_tr_wn_sc = sc_wn.fit_transform(X_tr_wn)
X_te_wn_sc = sc_wn.transform(X_te_wn)

print(f"Wine (binarised): {X_wn.shape[0]} samples, {X_wn.shape[1]} features")
print(f"Class counts — Barolo (1): {y_wn.sum()}, other (0): {(y_wn==0).sum()}")
print(f"Train/test split: {len(y_tr_wn)} / {len(y_te_wn)}")

---
## Part A — Sigmoid, cross-entropy, and gradient descent from scratch

We build logistic regression from first principles before using scikit-learn,
following the derivations in the lecture.

### A1. Sigmoid function

In [ ]:
def sigmoid(t):
    return 1.0 / (1.0 + np.exp(-t))

t_vals = np.linspace(-6, 6, 300)
plt.figure(figsize=(7, 3.5))
plt.plot(t_vals, sigmoid(t_vals), color="steelblue")
plt.axhline(0.5, color="grey", ls="--", lw=0.8)
plt.axvline(0.0, color="grey", ls="--", lw=0.8)
plt.xlabel("$t$"); plt.ylabel("$\sigma(t)$")
plt.title("Sigmoid function")
plt.tight_layout(); plt.show()

print(f"sigma(0)  = {sigmoid(0):.4f}  (should be 0.5)")
print(f"sigma(-3) = {sigmoid(-3):.4f},  1 - sigma(3) = {1 - sigmoid(3):.4f}  (symmetry)")
print(f"sigma'(0) = {sigmoid(0)*(1-sigmoid(0)):.4f}  (should be 0.25)")

### A2. Cross-entropy loss and gradient

In [ ]:
def cross_entropy_loss(y, p_hat):
    """Binary cross-entropy, averaged over n observations."""
    p_hat = np.clip(p_hat, 1e-12, 1 - 1e-12)
    return -np.mean(y * np.log(p_hat) + (1 - y) * np.log(1 - p_hat))

def gradient(X, y, w, b):
    """Gradient of cross-entropy w.r.t. w and b."""
    n = len(y)
    p = sigmoid(X @ w + b)
    err = p - y
    return X.T @ err / n, err.mean()

# Verify on a synthetic two-feature example
X_demo = np.array([[1.0, 2.0], [-1.0, -1.0], [2.0, -0.5], [-2.0, 1.0]])
y_demo = np.array([1, 0, 1, 0])
w_demo = np.zeros(2); b_demo = 0.0
p_demo = sigmoid(X_demo @ w_demo + b_demo)
print(f"Initial loss (all probs = 0.5): {cross_entropy_loss(y_demo, p_demo):.4f}")
print(f"Expected: {np.log(2):.4f}  (= log 2)")

### A3. Gradient descent loop

In [ ]:
def fit_logistic(X, y, lr=0.1, max_iter=1000, tol=1e-6):
    w = np.zeros(X.shape[1]); b = 0.0
    losses = []
    for i in range(max_iter):
        p = sigmoid(X @ w + b)
        losses.append(cross_entropy_loss(y, p))
        gw, gb = gradient(X, y, w, b)
        w -= lr * gw; b -= lr * gb
        if i > 0 and abs(losses[-1] - losses[-2]) < tol:
            print(f"  Converged at iteration {i+1}")
            break
    return w, b, losses

# Run on a synthetic linearly separable dataset
X_syn, y_syn = make_classification(
    n_samples=200, n_features=2, n_redundant=0, n_informative=2,
    random_state=1, n_clusters_per_class=1
)

# Split first, then fit the scaler on the training portion only
X_tr_s_raw, X_te_s_raw, y_tr_s, y_te_s = train_test_split(
    X_syn, y_syn, test_size=0.3, random_state=0
)
sc_syn = StandardScaler()
X_tr_s = sc_syn.fit_transform(X_tr_s_raw)
X_te_s = sc_syn.transform(X_te_s_raw)

w_slow, b_slow, losses_slow = fit_logistic(X_tr_s, y_tr_s, lr=0.01)
w_fast, b_fast, losses_fast = fit_logistic(X_tr_s, y_tr_s, lr=0.5)

plt.figure(figsize=(8, 4))
plt.plot(losses_slow, label="lr = 0.01 (slow)")
plt.plot(losses_fast, label="lr = 0.50 (fast)")
plt.xlabel("Iteration"); plt.ylabel("Cross-entropy loss")
plt.title("Gradient descent convergence")
plt.legend(); plt.tight_layout(); plt.show()

**In-class exercise.** The loss curve for a very large learning rate (e.g.\ $\eta = 2$)
sometimes oscillates or diverges rather than converging. Try `lr=2.0` and describe
what you observe. What condition on the learning rate is required for gradient descent
to converge?

---
## Part B — Decision boundaries

We visualise the linear boundary on separable data, then use polynomial feature
expansion to fit a non-linear boundary on the moons dataset.

### B1. Linear decision boundary

In [ ]:
fig, ax = plt.subplots(figsize=(6, 5))
h = 0.02
x1_min, x1_max = X_tr_s[:, 0].min() - 0.5, X_tr_s[:, 0].max() + 0.5
x2_min, x2_max = X_tr_s[:, 1].min() - 0.5, X_tr_s[:, 1].max() + 0.5
xx, yy = np.meshgrid(np.arange(x1_min, x1_max, h),
                     np.arange(x2_min, x2_max, h))
Z = sigmoid(np.c_[xx.ravel(), yy.ravel()] @ w_fast + b_fast).reshape(xx.shape)
ax.contourf(xx, yy, Z, levels=50, cmap="RdBu_r", alpha=0.6)
ax.contour(xx, yy, Z, levels=[0.5], colors="k", linewidths=1.5)
ax.scatter(X_tr_s[:, 0], X_tr_s[:, 1], c=y_tr_s,
           cmap="bwr", edgecolors="k", s=25, linewidths=0.4)
ax.set_title("Linear decision boundary (probability colour map)")
ax.set_xlabel("Feature 1"); ax.set_ylabel("Feature 2")
plt.tight_layout(); plt.show()

### B2. Non-linear boundary with polynomial features

In [ ]:
from sklearn.preprocessing import PolynomialFeatures

X_moon, y_moon = make_moons(n_samples=300, noise=0.25, random_state=0)
X_tr_m, X_te_m, y_tr_m, y_te_m = train_test_split(
    X_moon, y_moon, test_size=0.3, random_state=0
)

fig, axes = plt.subplots(1, 3, figsize=(14, 4))
degrees = [1, 2, 5]
x1_min, x1_max = X_moon[:, 0].min()-0.3, X_moon[:, 0].max()+0.3
x2_min, x2_max = X_moon[:, 1].min()-0.3, X_moon[:, 1].max()+0.3
xx, yy = np.meshgrid(np.arange(x1_min, x1_max, 0.02),
                     np.arange(x2_min, x2_max, 0.02))

for ax, deg in zip(axes, degrees):
    pipe = Pipeline([
        ("poly", PolynomialFeatures(degree=deg, include_bias=False)),
        ("scaler", StandardScaler()),
        ("clf", LogisticRegression(C=1.0, max_iter=2000))
    ])
    pipe.fit(X_tr_m, y_tr_m)
    Z = pipe.predict_proba(np.c_[xx.ravel(), yy.ravel()])[:, 1].reshape(xx.shape)
    ax.contourf(xx, yy, Z, levels=50, cmap="RdBu_r", alpha=0.6)
    ax.contour(xx, yy, Z, levels=[0.5], colors="k", linewidths=1.5)
    ax.scatter(X_tr_m[:, 0], X_tr_m[:, 1], c=y_tr_m,
               cmap="bwr", edgecolors="k", s=18, linewidths=0.4)
    tr_acc = accuracy_score(y_tr_m, pipe.predict(X_tr_m))
    te_acc = accuracy_score(y_te_m, pipe.predict(X_te_m))
    ax.set_title(f"degree={deg}  train={tr_acc:.2f}  test={te_acc:.2f}")
plt.suptitle("Polynomial logistic regression on moons", y=1.02)
plt.tight_layout(); plt.show()

**In-class exercise.** The degree-5 model achieves higher training accuracy than
degree 2 but similar or lower test accuracy. What does this suggest about the
bias--variance trade-off as polynomial degree increases? How does regularisation
(choice of C) interact with the choice of degree?

---
## Part C — Regularisation and the parameter C

We use the binarised Wine dataset (13 features, Barolo cultivar vs rest) to study
how the regularisation parameter C controls coefficient shrinkage and to select
C by cross-validation.

Recall from earlier in this lecture that C is the inverse of the penalty weight $\lambda$:
large C means weak regularisation, small C means strong regularisation.

### C1. Coefficient paths as C varies

In [ ]:
C_values = np.logspace(-3, 3, 100)
coef_paths = []

for C in C_values:
    clf = LogisticRegression(C=C, max_iter=2000, random_state=0)
    clf.fit(X_tr_wn_sc, y_tr_wn)
    coef_paths.append(clf.coef_[0].copy())

coef_paths = np.array(coef_paths)

fig, ax = plt.subplots(figsize=(11, 5))
for j, name in enumerate(wine.feature_names):
    ax.plot(C_values, coef_paths[:, j], lw=0.9, alpha=0.8, label=name)
ax.set_xscale("log")
ax.axvline(1.0, color="black", ls="--", lw=1, label="C = 1 (default)")
ax.set_xlabel("C (inverse regularisation strength)")
ax.set_ylabel("Coefficient value")
ax.set_title("Logistic regression coefficient paths (Wine, 13 features, Barolo vs rest)")
ax.legend(fontsize=7, ncol=2, loc="upper left")
plt.tight_layout(); plt.show()

At small C (left), all coefficients are pulled toward zero. As C increases
the model is free to fit the training data more closely and coefficients grow.
Some features receive large positive or negative coefficients — these are the
features most discriminative for the Barolo cultivar.

### C2. Choosing C by cross-validation

In [ ]:
C_grid = np.logspace(-3, 3, 40)
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=0)
cv_accs = []

for C in C_grid:
    scores = cross_val_score(
        LogisticRegression(C=C, max_iter=2000, random_state=0),
        X_tr_wn_sc, y_tr_wn, cv=skf, scoring="accuracy"
    )
    cv_accs.append(scores.mean())

cv_accs = np.array(cv_accs)
best_C = C_grid[cv_accs.argmax()]

plt.figure(figsize=(8, 4))
plt.plot(C_grid, cv_accs, color="steelblue", lw=1.5)
plt.axvline(best_C, color="firebrick", ls="--", lw=1,
            label=f"Best C = {best_C:.3f}")
plt.xscale("log")
plt.xlabel("C"); plt.ylabel("5-fold CV accuracy")
plt.title("Cross-validation accuracy vs C (Wine, Barolo vs rest)")
plt.legend(); plt.tight_layout(); plt.show()

print(f"Best C: {best_C:.4f}   CV accuracy: {cv_accs.max():.4f}")

clf_best = LogisticRegression(C=best_C, max_iter=2000, random_state=0)
clf_best.fit(X_tr_wn_sc, y_tr_wn)
print(f"Test accuracy with best C: {accuracy_score(y_te_wn, clf_best.predict(X_te_wn_sc)):.4f}")

---
## Part D — Classification metrics

Using the best model from Part C, we compute the full confusion matrix and all
five standard derived metrics, then examine when accuracy alone is misleading.

### D1. Confusion matrix and derived metrics

In [ ]:
y_pred_wn = clf_best.predict(X_te_wn_sc)
cm = confusion_matrix(y_te_wn, y_pred_wn)
tn, fp, fn, tp = cm.ravel()

print(f"Confusion matrix:")
print(f"  TN = {tn}   FP = {fp}")
print(f"  FN = {fn}   TP = {tp}")
print()

acc  = (tp + tn) / (tp + tn + fp + fn)
prec = tp / (tp + fp) if (tp + fp) > 0 else 0
rec  = tp / (tp + fn) if (tp + fn) > 0 else 0
spec = tn / (tn + fp) if (tn + fp) > 0 else 0
f1   = 2 * prec * rec / (prec + rec) if (prec + rec) > 0 else 0

print(f"Accuracy:    {acc:.4f}   (sklearn: {accuracy_score(y_te_wn, y_pred_wn):.4f})")
print(f"Precision:   {prec:.4f}   (sklearn: {precision_score(y_te_wn, y_pred_wn):.4f})")
print(f"Recall:      {rec:.4f}   (sklearn: {recall_score(y_te_wn, y_pred_wn):.4f})")
print(f"Specificity: {spec:.4f}")
print(f"F1 score:    {f1:.4f}   (sklearn: {f1_score(y_te_wn, y_pred_wn):.4f})")

In [ ]:
fig, ax = plt.subplots(figsize=(4.5, 4))
im = ax.imshow(cm, cmap="Blues")
ax.set_xticks([0, 1]); ax.set_yticks([0, 1])
ax.set_xticklabels(["Pred: other", "Pred: Barolo"])
ax.set_yticklabels(["True: other", "True: Barolo"])
for i in range(2):
    for j in range(2):
        ax.text(j, i, cm[i, j], ha="center", va="center",
                color="white" if cm[i, j] > cm.max() / 2 else "black", fontsize=14)
ax.set_title("Confusion matrix — Wine test set")
plt.colorbar(im, ax=ax)
plt.tight_layout(); plt.show()

### D2. When accuracy is misleading

In [ ]:
# Construct a strongly imbalanced synthetic dataset (95% negative)
X_imb, y_imb = make_classification(
    n_samples=1000, n_features=10, weights=[0.95, 0.05],
    random_state=2, flip_y=0.0
)
X_tr_i, X_te_i, y_tr_i, y_te_i = train_test_split(
    X_imb, y_imb, test_size=0.3, random_state=0, stratify=y_imb
)

# Majority-class dummy vs logistic regression
dummy = DummyClassifier(strategy="most_frequent")
dummy.fit(X_tr_i, y_tr_i)

lr_imb = LogisticRegression(max_iter=1000, random_state=0)
lr_imb.fit(X_tr_i, y_tr_i)

for name, clf in [("Majority-class dummy", dummy), ("Logistic regression", lr_imb)]:
    y_p = clf.predict(X_te_i)
    print(f"{name}:")
    print(f"  Accuracy  = {accuracy_score(y_te_i, y_p):.4f}")
    print(f"  Recall    = {recall_score(y_te_i, y_p):.4f}")
    print(f"  Precision = {precision_score(y_te_i, y_p, zero_division=0):.4f}")
    print(f"  F1        = {f1_score(y_te_i, y_p, zero_division=0):.4f}")
    print()

The majority-class dummy matches or beats logistic regression on accuracy while
achieving zero recall — it never identifies a single positive case. This is the
standard illustration of why accuracy is an unreliable metric under class imbalance.

**In-class exercise.** Using the Wine test set from Part D1, compute the precision
and recall manually from the confusion matrix entries and verify they match the
sklearn values. Then explain in one sentence why recall is a more important metric
than precision in a fault-detection context.

---
## Part E — ROC curves and AUC

We compare several models on the Wine binary task using predicted probabilities,
and examine the precision-recall curve alongside the ROC curve.

In [ ]:
# Fit comparison models on Wine
knn = Pipeline([("sc", StandardScaler()),
                ("clf", KNeighborsClassifier(n_neighbors=7))])
knn.fit(X_tr_wn, y_tr_wn)

lr_strong = LogisticRegression(C=0.001, max_iter=2000, random_state=0)
lr_strong.fit(X_tr_wn_sc, y_tr_wn)

dummy_roc = DummyClassifier(strategy="most_frequent")
dummy_roc.fit(X_tr_wn, y_tr_wn)

models = {
    "LR (best C)":   (clf_best, X_te_wn_sc),
    "LR (C=0.001)":  (lr_strong,  X_te_wn_sc),
    "k-NN (k=7)":    (knn,      X_te_wn),
    "Dummy":         (dummy_roc, X_te_wn),
}

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
for name, (model, X_te) in models.items():
    if hasattr(model, "predict_proba"):
        scores = model.predict_proba(X_te)[:, 1]
    else:
        scores = model.predict(X_te).astype(float)
    fpr, tpr, _ = roc_curve(y_te_wn, scores)
    auc = roc_auc_score(y_te_wn, scores)
    axes[0].plot(fpr, tpr, label=f"{name}  AUC={auc:.3f}")

axes[0].plot([0, 1], [0, 1], "k--", lw=0.8)
axes[0].set_xlabel("False positive rate"); axes[0].set_ylabel("True positive rate")
axes[0].set_title("ROC curves — Wine (Barolo vs rest)")
axes[0].legend(fontsize=9)

# Precision-recall curve for best model
probs_best = clf_best.predict_proba(X_te_wn_sc)[:, 1]
prec_curve, rec_curve, thresholds = precision_recall_curve(y_te_wn, probs_best)
axes[1].plot(rec_curve, prec_curve, color="steelblue")
axes[1].set_xlabel("Recall"); axes[1].set_ylabel("Precision")
axes[1].set_title("Precision-recall curve (best LR model)")

plt.tight_layout(); plt.show()

**In-class exercise.** The AUC of the majority-class dummy is 0.5, corresponding
to the diagonal of the ROC plot. Explain why this is the case from the definition
of AUC as the probability that the model ranks a random positive above a random
negative.

---
## Part F — Multiclass logistic regression (softmax)

We extend to three-class classification on Iris using the multinomial (softmax)
formulation, and examine per-class performance and probability outputs.

In [ ]:
iris = load_iris(as_frame=True)
X_ir, y_ir = iris.data.values, iris.target.values
X_tr_ir, X_te_ir, y_tr_ir, y_te_ir = train_test_split(
    X_ir, y_ir, test_size=0.3, random_state=0, stratify=y_ir
)
sc_ir = StandardScaler()
X_tr_ir_sc = sc_ir.fit_transform(X_tr_ir)
X_te_ir_sc = sc_ir.transform(X_te_ir)



In [ ]:
clf_soft = LogisticRegression(solver="lbfgs", max_iter=1000)
clf_soft.fit(X_tr_ir_sc, y_tr_ir)

print("Softmax logistic regression (Iris):")
print(classification_report(y_te_ir, clf_soft.predict(X_te_ir_sc),
                             target_names=iris.target_names))

# Verify probabilities sum to 1
probs = clf_soft.predict_proba(X_te_ir_sc[:5])
print("Predicted probabilities for first five test points:")
for i, row in enumerate(probs):
    print(f"  [{', '.join(f'{p:.4f}' for p in row)}]  sum = {row.sum():.6f}")

---
## Take-home exercises

**Exercise 1.** Using the Wine dataset (full three-class version, `load_wine()`),
fit a softmax logistic regression and a one-vs-rest logistic regression, each with
5-fold CV over $C \in \{0.001, 0.01, 0.1, 1, 10, 100\}$. Report the best C for each,
the cross-validated accuracy, and the macro-averaged F1 score on the test set.
Which formulation performs better on this dataset?

**Exercise 2.** Implement a threshold sweep on the best binarised Wine model from
Part E. For thresholds $t \in \{0.1, 0.2, \ldots, 0.9\}$, compute precision,
recall, and F1. Plot all three against $t$ and identify the threshold that maximises F1.
Report how many Barolo wines are misclassified as "other" at this optimal threshold.

**Exercise 3.** Fit logistic regression with $\ell_1$ regularisation
(`penalty='l1', solver='liblinear'`) on the binarised Wine data across the same
C grid as Part C2. Plot the coefficient paths and compare them to the $\ell_2$
paths from C1. How many coefficients are exactly zero at the CV-optimal C, and
which features survive?

**Exercise 4.** Using the Digits dataset (`load_digits()`), fit a softmax logistic
regression and report the full per-class classification report. Identify the two
digit classes with the lowest F1 score and display three misclassified examples from
each using `plt.imshow` on the $8 \times 8$ pixel array.